# Submission — pooled ensemble (seed 11 + seed 23), register-transfer BanglaT5

**Total inference parameters: 2 × 247,577,856 = 495,155,712 (0.495B)** — well under the 3B cap,
asserted at load.

| Model | Token F1 | ROUGE-L | pred LB |
|---|---|---|---|
| seed 11 | 0.7724 | 0.7324 | 0.8454 |
| seed 23 | 0.7723 | 0.7332 | 0.8455 |
| **pooled MBR** | **measured below** | | |

## ⚠️ Honest expectation: MBR may not win here

MBR draws **samples** (temp 0.8, top_p 0.95) from each model, pools them, and picks the candidate
with the highest expected utility against the others. It wins when a model is *uncertain* and its
samples scatter around the truth.

Two reasons that may not apply:
1. **The task is near-deterministic** — the answer is largely recoverable from the draft, and beam-4
   already reaches Token F1 0.772. Sampling injects noise into a problem with little left to hedge.
2. **The two seeds are near-identical** — 0.7724 vs 0.7723, a gap of 0.0001. Pooling two models that
   agree gives the selector almost nothing to choose between.

So this notebook **measures before it writes**, and ships whichever decoder actually won on dev.

## ⚠️ Before running
| Setting | Value |
|---|---|
| Accelerator | **GPU T4** |
| Internet | **On** |
| Inputs | competition · `didhitinahid/nascenia-code` · `didhitinahid/nascenia-drafts-devtest` · `didhitinahid/nascenia-xfer-ckpt` · `didhitinahid/nascenia-xfer-ckpt-s23` |

Runtime ~1.5–2 h (three dev configs, then the winning decoder over 1,000 test rows).

In [ ]:
# ══ 1 — hardware gate ═══════════════════════════════════════════════════════
import torch
n = torch.cuda.device_count()
assert n > 0, "No GPU. Settings -> Accelerator -> GPU T4"
cap = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__} | {[torch.cuda.get_device_name(i) for i in range(n)]} | sm_{cap[0]}{cap[1]}")
assert cap[0] >= 7, f"WRONG ACCELERATOR sm_{cap[0]}{cap[1]}"
print("✅ hardware OK")

In [ ]:
# ══ 2 — pinned libs (trap #00) ══════════════════════════════════════════════
!pip install -q --upgrade "transformers==4.57.3" git+https://github.com/csebuetnlp/normalizer
import transformers
assert transformers.__version__ == "4.57.3", transformers.__version__
from normalizer import normalize
print("transformers", transformers.__version__, "| normalizer OK:", normalize("হেলো,  নাসেনিয়া ডকে"))

In [ ]:
# ══ 3 — locate code, data, drafts, BOTH checkpoints ═════════════════════════
import glob, os, shutil, sys, subprocess
print("/kaggle/input:", os.listdir("/kaggle/input"))

hits = glob.glob("/kaggle/input/**/04_decode.py", recursive=True)
assert hits, "Attach didhitinahid/nascenia-code"
CODE = os.path.dirname(hits[0])
raw = glob.glob("/kaggle/input/**/test.csv", recursive=True)
assert raw, "Attach the competition"
RAW = os.path.dirname(raw[0])
dr = glob.glob("/kaggle/input/**/hcm_drafts_devtest.csv", recursive=True)
assert dr, "Attach didhitinahid/nascenia-drafts-devtest"
DRAFTS = dr[0]

# Pin each checkpoint by dataset name — a generic glob would not tell them apart,
# and the integrity check below is specific to seed 11.
def find_ckpt(name):
    c = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
         if name in p and glob.glob(os.path.dirname(p) + "/*.safetensors")]
    assert len(c) == 1, f"expected one checkpoint matching {name!r}, found {c}"
    return c[0]

CKPT11 = find_ckpt("nascenia-xfer-ckpt/")     # trailing slash: must not match ...-ckpt-s23
CKPT23 = find_ckpt("nascenia-xfer-ckpt-s23")
assert CKPT11 != CKPT23, "both names resolved to the same directory"

os.makedirs("/kaggle/working/code", exist_ok=True)
for f in glob.glob(f"{CODE}/*.py"):
    shutil.copy(f, "/kaggle/working/code/")
sys.path.insert(0, "/kaggle/working/code")
_h = subprocess.run(["python", "04_decode.py", "--help"], cwd="/kaggle/working/code",
                    capture_output=True, text=True).stdout
assert "--data-dir" in _h, "STALE CODE DATASET (trap #18) — re-attach newest nascenia-code"
print("✅ code current\nCKPT11:", CKPT11, "\nCKPT23:", CKPT23)

In [ ]:
# ══ 4 — frozen split (seed 42 governs the SPLIT — never change) ═════════════
!cd /kaggle/working/code && python 01_prep.py --raw "{RAW}" --out /kaggle/working/processed --seed 42 --dev-size 5000

In [ ]:
# ══ 5 — register-transfer inputs: the model sees the DRAFT ══════════════════
import pandas as pd, os
drafts = pd.read_csv(DRAFTS).drop_duplicates("hcm_id").set_index("hcm_id")["draft"]
os.makedirs("/kaggle/working/xfer", exist_ok=True)
for split in ("dev", "test"):
    df = pd.read_parquet(f"/kaggle/working/processed/{split}.parquet")
    missing = sorted(set(df["id"]) - set(drafts.index))
    assert not missing, f"{split}: {len(missing)} rows have no draft"
    out = pd.DataFrame({"id": df["id"].values, "input": drafts.loc[df["id"]].values})
    if "output" in df.columns:
        out["output"] = df["output"].values
    out.to_parquet(f"/kaggle/working/xfer/{split}.parquet", index=False)
    print(f"  {split:4s} {len(out):5d} rows · 100% draft coverage")

## 6 — Three decoders on the SAME 300 dev rows
seed-11 beam · seed-23 beam · pooled MBR. Like-for-like, so the comparison is real.

In [ ]:
# ══ 6 — dev bake-off ════════════════════════════════════════════════════════
import subprocess, shlex
BASE = "python 04_decode.py --split dev --limit 300 --data-dir /kaggle/working/xfer --no-bertscore"
BEAM = "--mode beam --num-beams 4 --min-new-tokens 80 --max-new-tokens 320 --length-penalty 1.0"

runs = [
    ("s11_beam", f"{BASE} --ckpt {shlex.quote(CKPT11)} {BEAM} --record /kaggle/working/d11.json"),
    ("s23_beam", f"{BASE} --ckpt {shlex.quote(CKPT23)} {BEAM} --record /kaggle/working/d23.json"),
    ("pooled_mbr", f"{BASE} --ckpt {shlex.quote(CKPT11)} --ckpt {shlex.quote(CKPT23)} "
                   f"--mode mbr -n 16 --temperature 0.8 --top-p 0.95 --min-new-tokens 80 "
                   f"--max-new-tokens 320 --record /kaggle/working/dmbr.json"),
]
for name, cmd in runs:
    print(f"\n=== {name}\n{cmd}", flush=True)
    r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code")
    assert r.returncode == 0, f"{name} failed"

In [ ]:
# ══ 7 — integrity check + pick the winner ═══════════════════════════════════
import json
def lex(p):
    d = json.load(open(p))["dev"]
    return d["token_f1"], d["rouge_l"], d["mean_pred_tokens"]

f11, r11, t11 = lex("/kaggle/working/d11.json")
f23, r23, t23 = lex("/kaggle/working/d23.json")
fmb, rmb, tmb = lex("/kaggle/working/dmbr.json")

# 🔴 The known-good-number gate. seed 11 beam-4 is a configuration with a recorded result,
# so if it does not reproduce, something is wrong with the weights, the code or the data —
# and nothing should be written. (This is what caught the fp16 NaN bug.)
assert abs(f11 - 0.7724) < 0.02, (
    f"CHECKPOINT MISMATCH: seed-11 beam expected ~0.7724, got {f11:.4f}. Do not submit.")
print("✅ seed-11 beam reproduced its recorded number — pipeline verified\n")

def lb(f, r): return 0.4672 + 0.3 * f + 0.2 * r
rows = [("seed 11 beam", f11, r11, t11), ("seed 23 beam", f23, r23, t23),
        ("pooled MBR", fmb, rmb, tmb)]
print(f"{'decoder':16s} {'TokenF1':>9s} {'ROUGE-L':>9s} {'tokens':>7s} {'pred LB':>9s}")
for nm, f, r, t in rows:
    print(f"{nm:16s} {f:9.4f} {r:9.4f} {t:7.1f} {lb(f,r):9.4f}")

best = max(rows, key=lambda x: lb(x[1], x[2]))
WINNER = best[0]
print(f"\n=> WINNER: {WINNER}  (pred LB {lb(best[1],best[2]):.4f})")

mbr_gain = lb(fmb, rmb) - max(lb(f11, r11), lb(f23, r23))
print(f"   MBR vs best single: {mbr_gain:+.4f} LB")
if mbr_gain > 0.002:
    print("   🥇 MBR genuinely helps — pooling adds signal")
elif mbr_gain > -0.002:
    print("   ➖ MBR is a wash — the two seeds agree too closely to give the selector anything")
else:
    print("   🔴 MBR LOSES — sampling noise costs more than consensus gains on this task")

## 8 — Generate the submission with the winning decoder

In [ ]:
# ══ 8 — test decode -> submission.csv ═══════════════════════════════════════
import subprocess, shlex
COMMON = ("python 04_decode.py --split test --data-dir /kaggle/working/xfer --no-bertscore "
          "--out /kaggle/working/submission.csv --record /kaggle/working/test_run.json "
          "--min-new-tokens 80 --max-new-tokens 320")
if WINNER == "pooled MBR":
    cmd = (f"{COMMON} --ckpt {shlex.quote(CKPT11)} --ckpt {shlex.quote(CKPT23)} "
           f"--mode mbr -n 16 --temperature 0.8 --top-p 0.95")
else:
    ck = CKPT11 if WINNER == "seed 11 beam" else CKPT23
    cmd = f"{COMMON} --ckpt {shlex.quote(ck)} --mode beam --num-beams 4 --length-penalty 1.0"
print(cmd + "\n" + "=" * 70, flush=True)
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code")
assert r.returncode == 0, "test decode failed"

In [ ]:
# ══ 9 — sanity checks + register read-out ═══════════════════════════════════
import pandas as pd, glob
sub = pd.read_csv("/kaggle/working/submission.csv")
test = pd.read_csv(glob.glob("/kaggle/input/**/test.csv", recursive=True)[0])
problems = []
if len(sub) != 1000:                       problems.append(f"expected 1000 rows, got {len(sub)}")
if list(sub.columns) != ["id", "output"]:  problems.append(f"columns are {list(sub.columns)}")
if sub["id"].duplicated().any():           problems.append("duplicate ids")
if set(sub["id"]) != set(test["id"]):      problems.append("id set does not match test.csv")
if sub["output"].isna().any():             problems.append("null outputs")
if (sub["output"].astype(str).str.strip() == "").any(): problems.append("empty outputs")

o = sub["output"].astype(str); L = o.str.split().str.len()
print(f"decoder used: {WINNER}")
print(f"rows {len(sub)} | mean {L.mean():.1f} tokens (refs ~100)")
print(f"opens হেলো   {o.str.startswith('হেলো').mean()*100:5.1f}%  (refs 76.4%, draft 0.1%)")
print(f"has নাসেনিয়া {o.str.contains('নাসেনিয়া').mean()*100:5.1f}%  (refs 50.0%, draft 0.0%)")
print(("\n❌ " + "; ".join(problems)) if problems else "\n✅ all checks passed — submission.csv ready")
sub.head(3)

---
## After this run

**Submit `submission.csv` from the Output tab yourself.** Record in `PREDICTIONS.md` which decoder
won and the MBR delta — that number is the first MBR measurement in this project and settles a
lever that has been open since day one, whichever way it goes.